In [0]:
from pyspark.sql.functions import expr, from_json, col, concat, lit, regexp_replace
from pyspark.sql.types import StructType, StructField, StringType, IntegerType

json_schema = StructType([
    StructField("country", StringType(), True),
    StructField("people_affected", IntegerType(), True),
    StructField("help_needed", StringType(), True),
    StructField("sector_code", StringType(), True)
])

categories_list = """
PRO: Protection (overall), FSC: Food Security, ALL: Final caseload, PRO-GBV: Gender-Based Violence, 
WSH: Water, Sanitation and Hygiene, HEA: Health, PRO-CPN: Child Protection, SHL: Emergency Shelter, 
NUT: Nutrition, EDU: Education, PRO-MIN: Mine Action, PRO-HLP: Housing, Land and Property, 
CCM: Camp Coordination And Camp Management, AGR: Agriculture, MPC: Multi-Purpose Cash, 
MS: Refugees and Migrant Multisector, CSS: Rapid Response Mechanism (RRM), 
ERY: Early Recovery and Livelihoods, LOG: Logistics, TEL: Emergency Telecommunications
"""

df = spark.table("unocha.default.reliefweb_articles_selected")

prompt_template = f"""
Extract data from the text. 
Task 1: Identify the PRIMARY affected country (as ISO3 code), the number of affected people (use 0 if unknown), and provide a short description of the help needed.
Task 2: Assign the article to ONE, most dominant category from this list: {categories_list}. Pick only ONE category.

Return the result ONLY as pure JSON, strictly in this exact format. Do NOT use markdown formatting, backticks, or any conversational text: 
{{"country": "X", "people_affected": 0, "help_needed": "Y", "sector_code": "CODE"}}

Text: 
"""

df_with_prompt = df.withColumn(
    "full_prompt", 
    concat(lit(prompt_template), regexp_replace(col("body"), "\n", " "))
)

enriched_df = df_with_prompt.withColumn(
    "llm_raw_response",
    expr("ai_query('databricks-meta-llama-3-1-8b-instruct', full_prompt)")
)


cleaned_json_col = regexp_replace(col("llm_raw_response"), r"(^```json\s+|\s+```$|^```\s+)", "")

final_parsed_df = enriched_df.withColumn(
    "extracted_data", 
    from_json(cleaned_json_col, json_schema)
)

final_df = final_parsed_df.select(
    "id",
    "title",
    "published_date",
    col("extracted_data.country").alias("country"),
    col("extracted_data.people_affected").alias("people_affected"),
    col("extracted_data.help_needed").alias("help_needed"),
    col("extracted_data.sector_code").alias("sector_code"),

)

final_df.write.mode("overwrite").saveAsTable("unocha.default.reliefweb_extracted")

print(f"Saved {final_df.count()} rows to unocha.default.reliefweb_extracted")
display(final_df)